# Assay: GPU health check on Kaggle

Builds Assay and its CUDA stress tests on the free GPU Kaggle lends you, runs a full
health check (about 5 minutes), verifies the signed report and packages it for download.

**Before you start:** in the notebook's right-hand panel, Session options → Accelerator → **GPU T4 x2** (two GPUs, so the all-reduce test runs too) or **GPU P100**, and Internet → **On** (needs a phone-verified Kaggle account). Then Run all.

Read-only: Assay never changes any GPU setting. It only allocates memory and runs its own tests.

## 1. Which GPU(s) did we get?
Each new session may give you a different physical GPU. The serial number tells them apart.

In [ ]:
!nvidia-smi --query-gpu=index,name,serial,uuid,compute_cap,driver_version,temperature.gpu --format=csv

## 2. Settings
`BRANCH`: the repo branch to build (`main` once your work is merged).
`SOAK`: how long the sustained-load test runs. 3 minutes is the standard; use `60s` for a quick try.

In [ ]:
REPO = "https://github.com/anushka11p/anushka11p-Assay-health-grades-for-GPU-backed-debt.git"
BRANCH = "main"
GO_VERSION = "1.27.3"   # must be >= the version in agent/go.mod
SOAK = "180s"
WORK = "/kaggle/working"     # where the report is saved
SRC = "/tmp/assay-src"

## 3. Install Go (skipped if already installed this session)

In [ ]:
%%bash -s "$GO_VERSION"
set -euo pipefail
if ! /tmp/go-sdk/go/bin/go version 2>/dev/null | grep -q "go$1 "; then
  rm -rf /tmp/go-sdk && mkdir -p /tmp/go-sdk
  curl -fsSL "https://go.dev/dl/go$1.linux-amd64.tar.gz" | tar -C /tmp/go-sdk -xz
fi
/tmp/go-sdk/go/bin/go version

## 4. Download Assay and build the agent

In [ ]:
%%bash -s "$REPO" "$BRANCH" "$SRC"
set -euo pipefail
rm -rf "$3"
git clone --quiet --depth 1 --branch "$2" "$1" "$3"
cd "$3/agent"
# NVIDIA's C headers print many harmless "deprecated" warnings; show only real errors.
if ! /tmp/go-sdk/go/bin/go build -o /tmp/assay ./cmd/assay 2>/tmp/build.log; then
  grep -v -e deprecated -e nvml.h -e cgo-gcc-prolog /tmp/build.log
  exit 1
fi
/tmp/assay version

## 5. Build the CUDA stress tests
Compiled for exactly this GPU's generation (e.g. `sm_75` for a T4, `sm_60` for a P100).
With 2+ GPUs it also builds the all-reduce test, fetching NVIDIA's NCCL library with pip if needed.

In [ ]:
%%bash -s "$SRC"
set -euo pipefail
export PATH=/usr/local/cuda/bin:$PATH
ARCH="sm_$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)"
echo "building probes for $ARCH"
make -s -C "$1/probes" ARCH="$ARCH"
GPUS=$(nvidia-smi -L | wc -l)
if [ "$GPUS" -ge 2 ]; then
  if ! make -s -C "$1/probes" allreduce ARCH="$ARCH" 2>/dev/null; then
    echo "system NCCL not found; installing it with pip"
    pip -q install nvidia-nccl-cu12
    NCCL_HOME=$(python3 -c "import nvidia.nccl as n; print(list(n.__path__)[0])")
    make -s -C "$1/probes" allreduce ARCH="$ARCH" NCCL_HOME="$NCCL_HOME" NCCL_LIB="-l:libnccl.so.2"
  fi
else
  echo "1 GPU: skipping the all-reduce test (needs 2+)"
fi
ls "$1/probes/bin"

## 6. Prove memtest can actually catch a bad bit
Plants one wrong word in GPU memory and checks memtest finds it. Expect `"mismatches":1` and `"self_test":"pass"`.

In [ ]:
!{SRC}/probes/bin/memtest --mb 256 --inject

## 7. Create a signing key for this session
The session is wiped when it ends, so each session gets a fresh key. Its public key is packaged with the report.

In [ ]:
!test -f /tmp/assay-keys/agent.key || /tmp/assay keygen --dir /tmp/assay-keys

## 8. Run the full health check (about 5 minutes)
1. 5 seconds of idle readings
2. Stress tests, in order: matrix multiply → memory bandwidth → memory pattern test → soak (+ all-reduce with 2+ GPUs)
3. Temperature, power, clocks and PCIe link sampled every 250 ms throughout

In [ ]:
import datetime, os
os.makedirs(WORK, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPORT = f"{WORK}/report-kaggle-{stamp}.json"
!/tmp/assay check --backend=nvml --key /tmp/assay-keys/agent.key --probe-dir {SRC}/probes/bin --soak {SOAK} --out {REPORT}

## 9. Verify the signature
Pinned to this session's public key: checks that the report is unmodified *and* who signed it.

In [ ]:
!/tmp/assay verify --pubkey /tmp/assay-keys/agent.pub {REPORT}

## 10. Package the report with its public key
The zip is saved in `/kaggle/working`: open the **Output** panel on the right and download it.
Unzip into `census/raw/` in your repo, keeping the `.json` and `.pub` together.

In [ ]:
import shutil, tempfile
pkg = tempfile.mkdtemp()
shutil.copy(REPORT, pkg)
shutil.copy("/tmp/assay-keys/agent.pub", REPORT.replace(".json", ".pub").replace(WORK, pkg))
ZIP = shutil.make_archive(REPORT[:-5], "zip", pkg)
print("packaged", ZIP)
print("download it from the Output panel →")